In [ ]:
import omero_analysis_notebook as oan
ctx = oan.configure(r'''{
  "schema": "nl.bioimaging.omero-analysis-notebook.v1",
  "inputs": [
    {"id": "measurements", "kind": "query", "path": "input/solhunt-measurements.duckdb", "formats": ["duckdb"], "required": true, "schema": {"tables": [{"name": "images"}, {"name": "channels"}, {"name": "objects"}, {"name": "intensity_features"}, {"name": "foci_assignments"}]}},
    {"id": "template", "kind": "file", "path": "input/template.xlsx", "extensions": [".xlsx"], "required": true}
  ],
  "results": {"path": "results"},
  "parameters": [
    {"name": "outlier_method", "type": "choice", "default": "MAD", "choices": ["MAD", "HistogramOtsu"], "label": "Outlier method", "help": "HistogramOtsu is a bounded 100-bin approximation and must be reviewed against the original full-vector result."},
    {"name": "mad_multiplier", "type": "number", "default": 4.0, "minimum": 0, "maximum": 100, "step": 1, "label": "MAD multiplier"},
    {"name": "marker_compartment", "type": "choice", "default": "cells", "choices": ["cells", "nuclei"], "label": "Measure"},
    {"name": "marker_channel", "type": "choice", "default": "3", "choices_query": {"source": "measurements", "sql": "SELECT DISTINCT CAST(channel_index AS VARCHAR) AS channel_index FROM channels ORDER BY channel_index LIMIT 100", "limit": 100, "value_column": "channel_index"}, "label": "Marker channel"}
  ],
  "requirements": ["numpy", "pandas", "matplotlib", "seaborn", "python-calamine"]
}''')
ctx.display_parameters()


## Portable conversion

The template remains a supporting XLSX input. Database access uses `await ctx.query`; no DuckDB connection or temporary registered relation is created. The template is validated and converted to a small SQL `VALUES` lookup. MAD remains exact. The former full CMP intensity transfer is replaced by 100 SQL histogram bins; `HistogramOtsu` is therefore explicitly approximate. Every plot is saved as PNG, SVG, and CSV under `ctx.results`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

raw = pd.read_excel(ctx.input("template"), header=None, engine="calamine")
columns = [str(int(value)) for value in raw.iloc[0, 1:] if pd.notna(value)]
rows = []
for value in raw.iloc[1:, 0]:
    expected = chr(65 + len(rows))
    if str(value).strip().upper() != expected:
        break
    rows.append(expected)
if (len(rows), len(columns)) not in {(16,24),(8,12),(6,8),(4,6),(3,4),(2,3)}:
    raise ValueError(f"Unsupported template dimensions: {len(rows)} x {len(columns)}")
template = raw.iloc[1:len(rows)+1, 1:len(columns)+1].copy()
template.index, template.columns = rows, columns
template_long = template.rename_axis("PlateRow").reset_index().melt(id_vars="PlateRow", var_name="PlateColumn", value_name="WellType")
template_long["SeriesName"] = template_long["PlateRow"] + template_long["PlateColumn"].astype(str).str.zfill(2)
template_long["WellType"] = template_long["WellType"].fillna("E").astype(str).str.strip()
if not set(template_long["WellType"]) <= {"CTL", "CMP", "E"}:
    raise ValueError("Template contains an unsupported well label")
if len(template_long) > 384:
    raise ValueError("Template lookup exceeds the bounded 384-well limit")
def literal(value):
    text = str(value)
    if not text.replace("_", "").isalnum():
        raise ValueError(f"Unsafe template value: {text}")
    return "'" + text.replace("'", "''") + "'"
template_values = ",".join(f"({literal(row.SeriesName)},{literal(row.WellType)})" for row in template_long.itertuples())
lookup = f"(VALUES {template_values}) AS tw(SeriesName, WellType)"


In [ ]:
params = {"compartment": ctx.params["marker_compartment"], "channel": int(ctx.params["marker_channel"])}
marker = f"""
SELECT i.plate_row || LPAD(i.plate_column, 2, '0') AS SeriesName, tw.WellType, inf.intensity_mean AS MarkerIntensity
FROM intensity_features inf
JOIN images i ON i.image_id = inf.image_id
JOIN {lookup} ON tw.SeriesName = i.plate_row || LPAD(i.plate_column, 2, '0')
WHERE inf.object_type = $compartment AND inf.channel_index = $channel
  AND tw.WellType IN ('CTL','CMP') AND inf.intensity_mean IS NOT NULL
"""
stats = await ctx.query("measurements", f"""
WITH marker AS ({marker}), base AS (
  SELECT COUNT(*) AS n, MEDIAN(MarkerIntensity) AS control_median
  FROM marker WHERE WellType='CTL'
)
SELECT base.n, base.control_median, MEDIAN(ABS(marker.MarkerIntensity-base.control_median)) AS control_mad,
       MIN(marker.MarkerIntensity) FILTER (WHERE marker.WellType='CMP') AS cmp_min,
       MAX(marker.MarkerIntensity) FILTER (WHERE marker.WellType='CMP') AS cmp_max
FROM marker CROSS JOIN base WHERE marker.WellType='CTL' GROUP BY base.n, base.control_median
""", params)
if stats.empty or int(stats.loc[0, "n"]) == 0:
    raise ValueError("No CTL marker intensities found")
control_median = float(stats.loc[0, "control_median"])
mad_threshold = control_median + float(ctx.params["mad_multiplier"]) * float(stats.loc[0, "control_mad"])
cmp_range = await ctx.query("measurements", f"WITH marker AS ({marker}) SELECT MIN(MarkerIntensity) AS lo, MAX(MarkerIntensity) AS hi FROM marker WHERE WellType='CMP'", params)
lo, hi = float(cmp_range.loc[0, "lo"]), float(cmp_range.loc[0, "hi"])
histogram = await ctx.query("measurements", f"""
WITH marker AS ({marker}), binned AS (
 SELECT WellType, LEAST(99, GREATEST(0, CAST(FLOOR((MarkerIntensity-$lo)/NULLIF($hi-$lo,0)*100) AS INTEGER))) AS BinIndex
 FROM marker WHERE MarkerIntensity BETWEEN $lo AND $hi
) SELECT WellType, BinIndex, COUNT(*) AS Count FROM binned GROUP BY WellType, BinIndex ORDER BY WellType, BinIndex
""", {**params, "lo": lo, "hi": hi})
cmp = histogram.loc[histogram.WellType.eq("CMP")].set_index("BinIndex")["Count"].reindex(range(100), fill_value=0)
centers = lo + (np.arange(100)+0.5)*(hi-lo)/100
weights = cmp.to_numpy(float)
probability = weights / weights.sum()
omega = np.cumsum(probability); mean = np.cumsum(probability*centers); total = mean[-1]
variance = (total*omega-mean)**2 / np.maximum(omega*(1-omega), np.finfo(float).eps)
histogram_otsu = float(centers[int(np.nanargmax(variance[:-1]))])
selected_threshold = mad_threshold if ctx.params["outlier_method"] == "MAD" else histogram_otsu
print({"control_median": control_median, "mad_threshold": mad_threshold, "histogram_otsu": histogram_otsu, "selected_threshold": selected_threshold})


In [ ]:
well_metrics = await ctx.query("measurements", f"""
WITH cell_map AS (
 SELECT c.object_id AS CellObjectID, CASE WHEN $compartment='cells' THEN c.object_id ELSE n.object_id END AS MarkerObjectID,
        i.plate_row || LPAD(i.plate_column,2,'0') AS SeriesName, tw.WellType
 FROM objects c JOIN images i ON i.image_id=c.image_id JOIN {lookup} ON tw.SeriesName=i.plate_row || LPAD(i.plate_column,2,'0')
 LEFT JOIN objects n ON n.image_id=c.image_id AND n.timepoint=c.timepoint AND n.label_value=c.label_value AND n.object_type='nuclei'
 WHERE c.object_type='cells' AND tw.WellType <> 'E'
), focus_counts AS (
 SELECT target_object_id AS CellObjectID, COUNT(DISTINCT source_object_id) AS NumGranules
 FROM foci_assignments WHERE target_object_type='cells' AND relation IN ('inside','identical_extent','overlaps') GROUP BY target_object_id
), values AS (
 SELECT cm.SeriesName, cm.WellType, COALESCE(fc.NumGranules,0) AS NumGranules, marker.intensity_mean AS MarkerIntensity
 FROM cell_map cm LEFT JOIN focus_counts fc ON fc.CellObjectID=cm.CellObjectID
 LEFT JOIN intensity_features marker ON marker.object_id=cm.MarkerObjectID AND marker.channel_index=$channel
)
SELECT SeriesName, WellType, COUNT(*) AS NumCells,
  100.0*AVG(CASE WHEN NumGranules>0 THEN 1.0 ELSE 0.0 END) AS PercentageCellsWithGranules,
  100.0*AVG(CASE WHEN MarkerIntensity>$threshold THEN 1.0 ELSE 0.0 END) FILTER (WHERE WellType='CMP') AS PercentageMarkerOutliers
FROM values GROUP BY SeriesName, WellType ORDER BY SeriesName
""", {"channel": int(ctx.params["marker_channel"]), "threshold": selected_threshold, "compartment": ctx.params["marker_compartment"]})
well_metrics


In [ ]:
def save_triplet(fig, stem, frame):
    fig.savefig(ctx.results / f"{stem}.png", bbox_inches="tight")
    fig.savefig(ctx.results / f"{stem}.svg", bbox_inches="tight")
    frame.to_csv(ctx.results / f"{stem}.csv", index=False)
    plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(11,5))
for group, color in (("CTL", "steelblue"), ("CMP", "darkorange")):
    frame = histogram[histogram.WellType.eq(group)]
    ax.plot(frame.BinIndex, frame.Count, label=group, color=color)
ax.set_yscale("log"); ax.set_xlabel("Shared intensity bin"); ax.set_ylabel("Count"); ax.legend()
save_triplet(fig, "marker_intensity_distributions", histogram)

plot_data = template_long[["SeriesName","PlateRow","PlateColumn","WellType"]].merge(well_metrics[["SeriesName","NumCells"]], on="SeriesName", how="left").fillna({"NumCells":0})
heat = plot_data.pivot(index="PlateRow", columns="PlateColumn", values="NumCells").reindex(index=rows, columns=columns)
fig, ax = plt.subplots(figsize=(20,8)); sns.heatmap(heat, annot=True, fmt=".0f", cmap="coolwarm", ax=ax); ax.set_title("Number of Cells")
save_triplet(fig, "number_of_cells", plot_data)
print("Portable SolHunt example complete; PNG/SVG/CSV triplets written.")
